### Cell 05.01 — load the frozen map and QTL results

In [ ]:
# Cell 05.01
# Load frozen structural map and final QTL results.

from pathlib import Path
import pandas as pd
import numpy as np


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()


MAP_FILE = (
    PROJECT_ROOT
    / "results"
    / "linkage_map"
    / "flyer_hartwig_structural_physical_map_final.xlsx"
)

QTL_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_all33_empirical_qtl_screen.xlsx"
)


print("MAP FILE EXISTS:", MAP_FILE.exists())
print("QTL FILE EXISTS:", QTL_FILE.exists())


map_xls = pd.ExcelFile(MAP_FILE)
qtl_xls = pd.ExcelFile(QTL_FILE)

print("\nMAP SHEETS")
print(map_xls.sheet_names)

print("\nQTL SHEETS")
print(qtl_xls.sheet_names)

### Cell 05.02 — load physical-anchor evidence and suggestive QTLs

In [ ]:
# Cell 05.02
# Load independent physical-anchor evidence
# and the final suggestive-QTL summary.

anchor_evidence = pd.read_excel(
    MAP_FILE,
    sheet_name="anchor_evidence"
)

ordered_framework = pd.read_excel(
    MAP_FILE,
    sheet_name="ordered_framework"
)

physical_assignments = pd.read_excel(
    MAP_FILE,
    sheet_name="physical_assignments"
)

all33_qtl = pd.read_excel(
    QTL_FILE,
    sheet_name="all_33_empirical"
)


suggestive_qtl = (
    all33_qtl
    .loc[
        all33_qtl[
            "genomewide_status"
        ] == "suggestive_10pct"
    ]
    .copy()
    .reset_index(drop=True)
)


print("ANCHOR EVIDENCE SHAPE:", anchor_evidence.shape)
print("ORDERED FRAMEWORK SHAPE:", ordered_framework.shape)
print("PHYSICAL ASSIGNMENTS SHAPE:", physical_assignments.shape)
print("SUGGESTIVE QTL:", len(suggestive_qtl))


print("\nANCHOR-EVIDENCE COLUMNS")
print(anchor_evidence.columns.tolist())

print("\nORDERED-FRAMEWORK COLUMNS")
print(ordered_framework.columns.tolist())

print("\nSUGGESTIVE QTL")
display(
    suggestive_qtl[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "lod",
            "peak_empirical_p"
        ]
    ]
)

### Cell 05.03 — inspect all anchor records for the four peak markers

In [ ]:
# Cell 05.03
# Find independent SoyBase/Wm82.gnm6 anchor evidence
# for the four suggestive peak markers.

peak_markers = (
    suggestive_qtl[
        "peak_marker"
    ]
    .dropna()
    .astype(str)
    .tolist()
)


print("PEAK MARKERS")
print(peak_markers)


# Search every object/string column for exact marker-name matches.
# This avoids assuming which exported column stores the marker ID.

object_cols = (
    anchor_evidence
    .select_dtypes(
        include=["object"]
    )
    .columns
    .tolist()
)


peak_anchor_mask = pd.Series(
    False,
    index=anchor_evidence.index
)


for col in object_cols:

    peak_anchor_mask |= (
        anchor_evidence[col]
        .astype(str)
        .isin(peak_markers)
    )


peak_anchor_evidence = (
    anchor_evidence
    .loc[
        peak_anchor_mask
    ]
    .copy()
)


print(
    "\nANCHOR RECORDS MATCHING "
    "SUGGESTIVE PEAK MARKERS"
)
print("=" * 140)

display(
    peak_anchor_evidence
)

### Cell 05.04 — inspect the complete local regions for physically anchored markers

In [ ]:
# Cell 05.04
# Examine physical evidence for ALL markers in the
# four suggestive linkage-fragment regions.

region_sheet_names = [
    "region_lrn",
    "region_scn_fi3",
    "region_prot_03",
    "region_days_fl_07"
]


region_markers = {}


for sheet in region_sheet_names:

    region = pd.read_excel(
        QTL_FILE,
        sheet_name=sheet
    )

    trait = sheet.replace(
        "region_",
        ""
    )

    markers = (
        region[
            "marker"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    region_markers[
        trait
    ] = markers


    mask = pd.Series(
        False,
        index=anchor_evidence.index
    )

    for col in object_cols:

        mask |= (
            anchor_evidence[col]
            .astype(str)
            .isin(markers)
        )


    hits = (
        anchor_evidence
        .loc[
            mask
        ]
        .copy()
    )


    print("\n")
    print("=" * 150)
    print(
        f"{trait}: "
        f"{len(markers)} markers in QTL fragment"
    )
    print("=" * 150)

    print("Markers:")
    print(markers)

    print(
        "\nIndependent physical-anchor records:"
    )

    display(hits)

### Cell 05.05 — summarize same-chromosome physical anchors for each suggestive locus

In [ ]:
# Cell 05.05
# Summarize independent physical anchors that agree
# with the candidate chromosome of each suggestive QTL.

qtl_anchor_rows = []


for _, qtl in suggestive_qtl.iterrows():

    trait = qtl["trait"]
    candidate_chr = qtl["candidate_chr"]

    region = pd.read_excel(
        QTL_FILE,
        sheet_name=f"region_{trait}"
    )

    region_marker_list = (
        region["marker"]
        .dropna()
        .astype(str)
        .tolist()
    )

    anchors = (
        anchor_evidence
        .loc[
            anchor_evidence["marker"].isin(
                region_marker_list
            )
        ]
        .copy()
    )

    anchors[
        "matches_candidate_chr"
    ] = (
        anchors["physical_chr"]
        == candidate_chr
    )

    anchors[
        "trait"
    ] = trait

    anchors[
        "candidate_chr"
    ] = candidate_chr

    qtl_anchor_rows.append(
        anchors
    )


qtl_region_anchor_evidence = (
    pd.concat(
        qtl_anchor_rows,
        ignore_index=True
    )
)


print(
    "PHYSICAL ANCHOR EVIDENCE FOR "
    "SUGGESTIVE QTL REGIONS"
)
print("=" * 150)

display(
    qtl_region_anchor_evidence[
        [
            "trait",
            "marker",
            "structural_group",
            "structural_order",
            "physical_chr",
            "physical_bp",
            "evidence_class",
            "matches_candidate_chr"
        ]
    ]
    .sort_values(
        [
            "trait",
            "structural_order"
        ]
    )
)

### Cell 05.06 — identify the nearest informative physical anchors around each peak
* Here "left" and "right" mean genetic-order sides, not physical upstream/downstream.

In [ ]:
# Cell 05.06
# Identify direct peak anchors and nearest concordant
# physical anchors on either side in genetic order.

localization_rows = []


for _, qtl in suggestive_qtl.iterrows():

    trait = qtl["trait"]
    peak_marker = qtl["peak_marker"]
    candidate_chr = qtl["candidate_chr"]

    region = pd.read_excel(
        QTL_FILE,
        sheet_name=f"region_{trait}"
    )

    peak_row = (
        region
        .loc[
            region["marker"]
            == peak_marker
        ]
        .iloc[0]
    )

    peak_order = int(
        peak_row["structural_order"]
    )


    anchors = (
        qtl_region_anchor_evidence
        .loc[
            (
                qtl_region_anchor_evidence[
                    "trait"
                ] == trait
            )
            &
            (
                qtl_region_anchor_evidence[
                    "physical_chr"
                ] == candidate_chr
            )
        ]
        .sort_values(
            "structural_order"
        )
        .copy()
    )


    direct = (
        anchors
        .loc[
            anchors["marker"]
            == peak_marker
        ]
    )

    left = (
        anchors
        .loc[
            anchors[
                "structural_order"
            ] < peak_order
        ]
        .tail(1)
    )

    right = (
        anchors
        .loc[
            anchors[
                "structural_order"
            ] > peak_order
        ]
        .head(1)
    )


    def get_value(
        df,
        col
    ):
        if len(df) == 0:
            return np.nan
        return df.iloc[0][col]


    localization_rows.append({

        "trait": trait,
        "peak_marker": peak_marker,
        "structural_group":
            qtl["structural_group"],
        "candidate_chr":
            candidate_chr,

        "peak_structural_order":
            peak_order,

        "direct_peak_anchor":
            len(direct) > 0,

        "direct_peak_bp":
            get_value(
                direct,
                "physical_bp"
            ),

        "direct_peak_evidence":
            get_value(
                direct,
                "evidence_class"
            ),

        "left_anchor_marker":
            get_value(
                left,
                "marker"
            ),

        "left_anchor_bp":
            get_value(
                left,
                "physical_bp"
            ),

        "left_anchor_evidence":
            get_value(
                left,
                "evidence_class"
            ),

        "right_anchor_marker":
            get_value(
                right,
                "marker"
            ),

        "right_anchor_bp":
            get_value(
                right,
                "physical_bp"
            ),

        "right_anchor_evidence":
            get_value(
                right,
                "evidence_class"
            )
    })


suggestive_qtl_localization = (
    pd.DataFrame(
        localization_rows
    )
)


display(
    suggestive_qtl_localization
)

### Cell 05.07 — define conservative physical localization classes
* This deliberately distinguishes a physical anchor bracket from a formal QTL confidence interval.

In [ ]:
# Cell 05.07
# Create conservative physical-localization summaries.
#
# IMPORTANT:
# Physical brackets are anchor-defined genomic ranges,
# NOT statistical QTL confidence intervals.

physical_summary_rows = []


for _, row in (
    suggestive_qtl_localization
    .iterrows()
):

    left_bp = row[
        "left_anchor_bp"
    ]

    right_bp = row[
        "right_anchor_bp"
    ]

    direct_bp = row[
        "direct_peak_bp"
    ]


    has_left = pd.notna(
        left_bp
    )

    has_right = pd.notna(
        right_bp
    )

    has_direct = pd.notna(
        direct_bp
    )


    if has_left and has_right:

        bracket_start = min(
            left_bp,
            right_bp
        )

        bracket_end = max(
            left_bp,
            right_bp
        )

        localization_class = (
            "two_sided_anchor_bracket"
        )

    elif has_direct:

        bracket_start = np.nan
        bracket_end = np.nan

        localization_class = (
            "direct_peak_anchor_no_closed_bracket"
        )

    elif has_left or has_right:

        bracket_start = np.nan
        bracket_end = np.nan

        localization_class = (
            "one_sided_anchor_only"
        )

    else:

        bracket_start = np.nan
        bracket_end = np.nan

        localization_class = (
            "no_physical_localization"
        )


    physical_summary_rows.append({

        "trait":
            row["trait"],

        "peak_marker":
            row["peak_marker"],

        "structural_group":
            row["structural_group"],

        "candidate_chr":
            row["candidate_chr"],

        "direct_peak_anchor":
            row["direct_peak_anchor"],

        "direct_peak_bp":
            direct_bp,

        "direct_peak_evidence":
            row[
                "direct_peak_evidence"
            ],

        "left_anchor_marker":
            row[
                "left_anchor_marker"
            ],

        "left_anchor_bp":
            left_bp,

        "right_anchor_marker":
            row[
                "right_anchor_marker"
            ],

        "right_anchor_bp":
            right_bp,

        "physical_bracket_start_bp":
            bracket_start,

        "physical_bracket_end_bp":
            bracket_end,

        "physical_bracket_span_mb":
            (
                (
                    bracket_end
                    - bracket_start
                ) / 1e6
                if pd.notna(
                    bracket_start
                )
                and pd.notna(
                    bracket_end
                )
                else np.nan
            ),

        "localization_class":
            localization_class
    })


suggestive_qtl_physical_summary = (
    pd.DataFrame(
        physical_summary_rows
    )
)


print(
    "CONSERVATIVE PHYSICAL LOCALIZATION"
)
print("=" * 160)

display(
    suggestive_qtl_physical_summary
)

### Cell 05.08 — explicitly incorporate chromosome-assignment confidence
* This is particularly important for days_fl_07.

In [ ]:
# Cell 05.08
# Join structural-group chromosome assignment confidence
# to the QTL physical-localization table.

assignment_cols = [
    "structural_group",
    "dominant_chr",
    "assignment_status"
]


assignment_lookup = (
    physical_assignments[
        assignment_cols
    ]
    .drop_duplicates(
        subset="structural_group"
    )
)


suggestive_qtl_physical_summary = (
    suggestive_qtl_physical_summary
    .merge(
        assignment_lookup,
        on="structural_group",
        how="left",
        validate="many_to_one"
    )
)


# Flag whether a locus is suitable for physical
# candidate-region interpretation.

def candidate_region_readiness(row):

    status = row[
        "assignment_status"
    ]

    localization = row[
        "localization_class"
    ]

    if (
        status
        == "insufficient_single_anchor"
    ):
        return "not_ready_single_anchor"

    if (
        localization
        == "two_sided_anchor_bracket"
    ):
        return "ready_broad_physical_bracket"

    if (
        localization
        == "direct_peak_anchor_no_closed_bracket"
    ):
        return "ready_peak_localization_only"

    return "limited_physical_resolution"


suggestive_qtl_physical_summary[
    "candidate_region_readiness"
] = (
    suggestive_qtl_physical_summary
    .apply(
        candidate_region_readiness,
        axis=1
    )
)


print(
    "QTL PHYSICAL LOCALIZATION + "
    "ANCHORING CONFIDENCE"
)
print("=" * 180)

display(
    suggestive_qtl_physical_summary[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "assignment_status",
            "direct_peak_bp",
            "physical_bracket_start_bp",
            "physical_bracket_end_bp",
            "physical_bracket_span_mb",
            "localization_class",
            "candidate_region_readiness"
        ]
    ]
)

### Cell 05.09 — create manuscript-ready physical localization summary

In [ ]:
# Cell 05.09
# Build a clean manuscript-ready physical localization table.

physical_localization_final = (
    suggestive_qtl_physical_summary
    .copy()
)


# Convert base-pair coordinates to Mb for readability.
for col in [
    "direct_peak_bp",
    "physical_bracket_start_bp",
    "physical_bracket_end_bp"
]:
    physical_localization_final[
        col.replace("_bp", "_mb")
    ] = (
        physical_localization_final[col]
        / 1e6
    )


# Add concise interpretation text.
def localization_interpretation(row):

    trait = row["trait"]

    if trait == "scn_fi3":
        return (
            "Two-sided exact-anchor bracket; "
            "best physically resolved suggestive QTL"
        )

    if trait == "lrn":
        return (
            "Broad two-sided anchor bracket; "
            "peak has assay-family physical evidence"
        )

    if trait == "prot_03":
        return (
            "Peak marker directly anchored; "
            "no closed physical bracket"
        )

    if trait == "days_fl_07":
        return (
            "Single-anchor chromosome correspondence only; "
            "no defensible physical interval"
        )

    return "limited physical resolution"


physical_localization_final[
    "interpretation"
] = (
    physical_localization_final
    .apply(
        localization_interpretation,
        axis=1
    )
)


display(
    physical_localization_final[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "assignment_status",
            "direct_peak_mb",
            "direct_peak_evidence",
            "physical_bracket_start_mb",
            "physical_bracket_end_mb",
            "physical_bracket_span_mb",
            "localization_class",
            "candidate_region_readiness",
            "interpretation"
        ]
    ]
)

### Cell 05.10 — make a candidate-region table for downstream gene searches
* This should include only loci where a physical region can be defended.

In [ ]:
# Cell 05.10
# Define physically usable regions for downstream candidate-gene work.

candidate_region_rows = []


for _, row in (
    physical_localization_final
    .iterrows()
):

    if (
        row["candidate_region_readiness"]
        == "ready_broad_physical_bracket"
    ):

        candidate_region_rows.append({
            "trait": row["trait"],
            "peak_marker": row["peak_marker"],
            "chromosome": row["candidate_chr"],
            "region_start_bp":
                int(
                    row[
                        "physical_bracket_start_bp"
                    ]
                ),
            "region_end_bp":
                int(
                    row[
                        "physical_bracket_end_bp"
                    ]
                ),
            "region_start_mb":
                row[
                    "physical_bracket_start_mb"
                ],
            "region_end_mb":
                row[
                    "physical_bracket_end_mb"
                ],
            "region_span_mb":
                row[
                    "physical_bracket_span_mb"
                ],
            "region_type":
                "anchor_defined_bracket"
        })


candidate_regions = (
    pd.DataFrame(
        candidate_region_rows
    )
)


print(
    "PHYSICALLY DEFENSIBLE CANDIDATE REGIONS"
)
print("=" * 140)

display(
    candidate_regions
)

### Cell 05.11 — create a separate peak-only localization table
* This preserves prot_03 without pretending it has a closed interval.

In [ ]:
# Cell 05.11
# Preserve loci with useful direct physical localization
# but no closed physical interval.

peak_only_localizations = (
    physical_localization_final
    .loc[
        physical_localization_final[
            "localization_class"
        ]
        ==
        "direct_peak_anchor_no_closed_bracket"
    ]
    [
        [
            "trait",
            "peak_marker",
            "candidate_chr",
            "direct_peak_bp",
            "direct_peak_mb",
            "direct_peak_evidence",
            "assignment_status",
            "interpretation"
        ]
    ]
    .copy()
)


print(
    "DIRECT PEAK LOCALIZATIONS WITHOUT "
    "CLOSED PHYSICAL INTERVALS"
)
print("=" * 140)

display(
    peak_only_localizations
)

### Cell 05.12 — save the physical-localization checkpoint

In [ ]:
# Cell 05.12
# Save physical localization results for downstream
# literature comparison and candidate-gene analysis.

QTL_PHYSICAL_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_qtl_physical_localization.xlsx"
)


with pd.ExcelWriter(
    QTL_PHYSICAL_FILE,
    engine="openpyxl"
) as writer:

    physical_localization_final.to_excel(
        writer,
        sheet_name="qtl_physical_summary",
        index=False
    )

    candidate_regions.to_excel(
        writer,
        sheet_name="candidate_regions",
        index=False
    )

    peak_only_localizations.to_excel(
        writer,
        sheet_name="peak_only_localization",
        index=False
    )

    qtl_region_anchor_evidence.to_excel(
        writer,
        sheet_name="anchor_evidence",
        index=False
    )


print(
    "Saved QTL physical localization:"
)

print(
    QTL_PHYSICAL_FILE
)